# Train only the TargetHead — stage on top of the pair+frac ckpt

**Only the TargetHead trains.** Everything else (4 encoders, pair head, frac head) is frozen so the existing val_top1 = 0.450 pair signal and val_frac_mae = 0.035 frac signal are preserved bit-exact. The pair loss is excluded from the total (``pair_weight=0``) so its gradients don't leak into the encoders either — the only training signal is ``target_loss`` driving ~17 k TargetHead params.

**Default starting ckpt**: `gs://orbit-wars-shipping/runs/exp3_frac_only_Ebi_20260511-150610/pair_score_best.pt` (pair val_top1 = 0.450, frac val_mae = 0.035). The combined ckpt this notebook writes adds a `target_head` state so a downstream PPO bring-up or runtime decoder can use all three heads at once.

**Local-CPU feasible**: with ~17 k trainable params and frozen encoders, the target stage runs in a few minutes per epoch on CPU at full Ebi (~32 k acted rows). For an iterative loop on a laptop, drop `max_rows` to 5 000 — wraps up in <10 min for 20 epochs.

**Prerequisites in `gs://orbit-wars-shipping/`**: `code.tgz`, `data.tgz`, `weights.tgz`, `pair_score_assets.tgz`. Build & upload locally with:

```bash
PAIR_SCORE_PLAYER=Ebi INCLUDE_PAIR_SCORE_ASSETS=1 UPLOAD=1 ./scripts/pack_for_gpu.sh
```

**Runtime**: Runtime → Change runtime type → T4 GPU (or run locally on CPU — see timing estimate above).

## 1. Verify GPU (or note CPU fallback)

In [ ]:
import torch, sys
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    DEVICE = 'cuda'
else:
    print('No GPU runtime — falling back to CPU. Target-only training is '
          'CPU-feasible since only ~17k params train; expect a few min/epoch.')
    DEVICE = 'cpu'
print(f'CUDA: {torch.version.cuda}  PyTorch: {torch.__version__}')

## 2. Configuration

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT = 'analog-receiver-489214-e9'
BUCKET  = 'gs://orbit-wars-shipping'
PLAYER  = 'Ebi'

# Stage-2 frac+pair ckpt we resume from. Edit if you have a newer one.
PRIOR_RUN = 'exp3_frac_only_Ebi_20260511-150610'
PRIOR_BEST_GCS = f'{BUCKET}/runs/{PRIOR_RUN}/pair_score_best.pt'

# Target-only training: pair head + frac head + all encoders frozen,
# pair_loss excluded from total. The TargetHead is the only thing that
# can change between epochs.
TARGET_WEIGHT     = 1.0
PAIR_WEIGHT       = 0.0       # exclude pair_loss from total
FRAC_WEIGHT       = 0.0       # frac head loaded + frozen, not trained
FREEZE_PAIR_HEAD  = True
FREEZE_FRAC_HEAD  = True
FREEZE_TARGET_HEAD = False    # the one head that trains
UNFREEZE          = ''        # no encoder thawing — preserve pair val_top1
LR                = 1e-3      # only a small head trains, can afford high LR
EPOCHS            = 25
BATCH_SIZE        = 64
VAL_FRAC          = 0.2
MAX_ROWS          = None      # None = full Ebi; set to 5000 for a fast laptop run

!gcloud config set project {PROJECT}

## 3. Pull tarballs + prior best.pt

In [ ]:
import os
WORK = '/content/orbit-wars'
os.makedirs(WORK, exist_ok=True)
%cd {WORK}

for name in ('code.tgz', 'data.tgz', 'weights.tgz', 'pair_score_assets.tgz'):
    !gsutil cp {BUCKET}/{name} .

PRIOR_BEST_LOCAL = f'data/runs/pair_score/{PRIOR_RUN}/pair_score_best.pt'
os.makedirs(os.path.dirname(PRIOR_BEST_LOCAL), exist_ok=True)
!gsutil cp {PRIOR_BEST_GCS} {PRIOR_BEST_LOCAL}

## 4. Unpack + validate

In [ ]:
%cd {WORK}
!tar xzf code.tgz
!tar xzf data.tgz
!tar xzf weights.tgz
!tar xzf pair_score_assets.tgz

import glob
from pathlib import Path

required = (
    ('data/datasets/action', 'action_*.csv', 'action CSVs'),
    ('data/datasets/planet', 'planet_*.csv', 'planet CSVs'),
    ('data/datasets/fleet', 'fleet_*.csv', 'fleet CSVs'),
    ('data/datasets/entity', 'entity_*.csv', 'entity CSVs'),
    ('data/datasets/cross_entity', 'cross_entity_*.csv', 'cross-entity CSVs'),
)
for rel, pattern, label in required:
    count = len(list(Path(rel).glob(pattern)))
    print(f'{label}: {count}')
    if count == 0:
        raise SystemExit(f'no {label} found under {rel}; data.tgz is stale.')

act = sorted(glob.glob('data/runs/action/*/action_best.pt'))
if not act:
    raise SystemExit('no action_*.pt under data/runs/action/.')
ENCODER_CKPT = act[-1]
print('encoder ckpt:', ENCODER_CKPT)

player_replays = sorted(glob.glob(f'data/replays/{PLAYER}/*.json.gz'))
if not player_replays:
    raise SystemExit(f'no replays under data/replays/{PLAYER}/.')
print(f'replays for {PLAYER}: {len(player_replays)}')

if not Path(PRIOR_BEST_LOCAL).exists():
    raise SystemExit(f'prior best.pt missing at {PRIOR_BEST_LOCAL}; section 3 failed.')
print('prior best.pt:', PRIOR_BEST_LOCAL)

## 5. Install + import

In [ ]:
%cd {WORK}
!pip install -q -r requirements.txt --no-deps
!pip install -q kaggle-environments

In [ ]:
import sys
sys.path.insert(0, WORK)

from agents.transformer_v1.pretrain.pair_score import (
    prepare_dataset, train_pair_score_kwargs,
)
print('imports OK — target-only training ready')

### 5b. Materialize the dataset (run once per session)

Parses Ebi's action CSVs into snapshot tensors and holds them in the kernel variable `dataset`. Every later training cell skips the parse and starts immediately.

If you regenerate action CSVs (e.g. after a featurizer change), restart this cell.

In [ ]:
dataset = prepare_dataset(
    player=PLAYER,
    filter_mode='all',
    max_planets=64,
    max_fleets=256,
    n_history=3,
    cache_dir=None,
    rebuild_cache=False,
)
print(f'dataset ready: {len(dataset)} snapshots held in kernel')

## 6. Train ONLY the TargetHead

Calls `train_pair_score_kwargs(...)` in this kernel — every epoch's `tr_tgt_top1 / val_tgt_top1 / val_tgt_top3` line streams here as it happens. The frozen pair head's `val_top1` should be **identical every epoch** since neither it nor the encoders are training.

**Realistic timing** (CPU = single Apple-M core; GPU = Colab T4):

| `MAX_ROWS` | rows/epoch | time/epoch (CPU) | time/epoch (T4 GPU) | 25-epoch total (T4) |
|---:|---:|---:|---:|---:|
| 1 000 | 800 train | ~45 s | ~5 s | ~2 min |
| 5 000 | 4 000 train | ~3.5 min | ~25 s | ~10 min |
| None (full Ebi, ~26 k) | ~26 000 train | ~25 min | ~3 min | ~75 min |

The encoder forward (frozen) dominates per-batch cost; only ~17 k TargetHead params back-propagate. Local CPU is fine for the 5 000-row iteration loop; T4 GPU recommended for full-Ebi runs.

In [ ]:
import time
TS = time.strftime('%Y%m%d-%H%M%S')
OUT_DIR = f'data/runs/pair_score/target_only_{PLAYER}_{TS}'
print('out dir:', OUT_DIR)

best_ckpt = train_pair_score_kwargs(
    encoder_ckpt=ENCODER_CKPT,
    out_dir=OUT_DIR,
    init_from=PRIOR_BEST_LOCAL,
    pair_weight=PAIR_WEIGHT,
    frac_weight=FRAC_WEIGHT,
    target_weight=TARGET_WEIGHT,
    freeze_pair_head=FREEZE_PAIR_HEAD,
    freeze_frac_head=FREEZE_FRAC_HEAD,
    freeze_target_head=FREEZE_TARGET_HEAD,
    unfreeze=(UNFREEZE if UNFREEZE else None),
    player=PLAYER,
    filter='all',
    max_rows=MAX_ROWS,
    val_frac=VAL_FRAC,
    batch_size=BATCH_SIZE,
    lr=LR,
    epochs=EPOCHS,
    device=DEVICE,
    dataset=dataset,
)
print('done. best ckpt:', best_ckpt)

import torch
ck = torch.load(best_ckpt, map_location='cpu', weights_only=False)
print('ckpt keys:', sorted(k for k in ck if not k.startswith('_')))

## 7. Log summary

In [ ]:
import json
log = json.loads(open(f'{WORK}/{OUT_DIR}/log.json').read())
for e in log:
    v = e['val']
    print(
        f"ep {e['epoch']:2d}  "
        f"tr_tgt_top1={e['train'].get('target_top1', 0):.3f}  "
        f"val_tgt_top1={v.get('target_top1', 0):.3f}  "
        f"val_tgt_top3={v.get('target_top3', 0):.3f}  "
        f"val_tgt_top5={v.get('target_top5', 0):.3f}  "
        f"||  pair_top1={v['top1']:.3f}  (frozen, sanity)"
    )
best = max(log, key=lambda e: e['val'].get('target_top1', 0))
print(f"\nbest val_tgt_top1={best['val'].get('target_top1', 0):.3f} (epoch {best['epoch']})")

## 8. Push results to GCS

In [ ]:
%cd {WORK}
!gsutil -m cp -r {OUT_DIR} {BUCKET}/runs/
print(f'uploaded {BUCKET}/runs/{OUT_DIR.rsplit("/", 1)[-1]}')